In [0]:
from pyspark.sql.functions import *

In [0]:
# =========================================================
# 1. SLV MONTH temp 로드
# =========================================================
fact_df = spark.table(
    "hive_metastore.demo_airstatus_silver.SLV_temp_fact_air_quality_month"
)

In [0]:
# =========================================================
# 2. 관측소 메타 (dmX / dmY 타입 강제 일치)
# =========================================================
stations_df = spark.table(
    "hive_metastore.demo_airstatus_bronze.BRZ_seoul_stations"
).select(
    "stationName",
    col("dmX").cast("double").alias("dmX"),   # ✅ 핵심
    col("dmY").cast("double").alias("dmY"),   # ✅ 핵심
    "addr"
)

In [0]:

# =========================================================
# 3. Geo 정보 조인
# =========================================================
fact_geo_df = (
    fact_df
    .join(
        stations_df,
        on="stationName",
        how="left"
    )
)

In [0]:
# =========================================================
# 4. Gold 테이블 append (기존 데이터 유지)
# =========================================================
fact_geo_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable(
        "hive_metastore.demo_airstatus_gold.GLD_fact_air_quality_dashboard"
    )

print("✅ GLD_MONTH data appended successfully.")

✅ GLD_MONTH data appended successfully.
